# 06 · Puerta de tamaño de muestra, clasificación y reporte — Muerte fetal, MIMIC-IV (M3 · R3)

| Campo | Contenido |
|---|---|
| Secuencia del framework | 01: E1 · E2 · E3 · P1 → 02: E4 → 03: E5 · E6 → R3 → 04: R3 · E2 · E3 · P1 · E4 → 05: E5 · E6 · E7 · P2 → **06: P3 · clase · E11** |
| Qué hace | Aplica la puerta P3 con el R² de Cox-Snell observado de una regresión logística con validación cruzada (particiones agrupadas por paciente; imputación y escalado ajustados con cada parte de entrenamiento) y los criterios de Riley et al.; solo se calcula el R² (no se mira ni el AUROC ni los coeficientes). Con el resultado, fija la clase del modelo y lo que se reporta (E11) |
| Etapas y puertas | P3 · clase · E11 |
| Decisiones | Clase del M3; el conjunto reducido no se aplica (solo se permite para un modelo que ya cumple P4) |
| Entradas | `antenatal_predictors_v1` (notebook 05), margen de 7 días |
| Salidas | `data/outputs_06_sample_size_gate.txt` (solo salida agregada) |
| Tiempo | Alrededor de un minuto |
| Anterior / siguiente | `05_antenatal_predictors` / — |
| Versión en inglés | `06_sample_size_gate.ipynb` (mismo código) |

In [1]:
# Setup: paths, language, presentation helpers and the database connection
import sys
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
# Language of the outputs: "es" in the Spanish notebook, "en" in the English one (the only line that differs).
LANG = "es"
import importlib
import libraries.i18n_es, libraries.i18n, libraries.presentation, libraries.sql_runner
for _mod in (libraries.i18n_es, libraries.i18n, libraries.presentation, libraries.sql_runner):
    importlib.reload(_mod)                       # a running kernel picks up changes of the libraries
from libraries.presentation import setup, tr, say, show, show_results, check, decision, notebook_summary, write_outputs
from libraries.sql_runner import run_script, ensure_table, table_exists
from libraries.database import get_connect_database
setup(LANG)

REBUILD = False      # True: rebuild the mimiciv tables of this notebook from their SQL scripts
SQL = ROOT / "sql"
OUT_FILE = ROOT / "data" / "outputs_06_sample_size_gate.txt"
engine = get_connect_database()
from libraries.antenatal_design import design_matrix, REDUCED_PARAMETERS
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)  # 'penalty' renamed in recent scikit-learn

SEED = 20261119
BUFFER_DAYS = 7          # main information window
N_FOLDS = 5
S_TARGET = 0.9           # Riley criterion 1: expected shrinkage
DELTA = 0.05             # Riley criterion 2: optimism in Nagelkerke R2
MARGIN = 0.05            # Riley criterion 3: precision of the overall risk

## P3 · Puerta de tamaño de muestra — datos (P1: misma población y mismos eventos)

El margen de 7 días de la tabla de predictores. El notebook se detiene si la población o los eventos difieren de la puerta P1.

| Resultado | Qué se hace | En este modelo |
|---|---|---|
| 19.440 admisiones y 129 eventos | Se pasa a la matriz de diseño | Reproducido |
| Difieren | Se detiene (`assert`): volver a los notebooks 04 y 05 | — |

In [2]:
df = pd.read_sql(
    "SELECT * FROM mimiciv.antenatal_predictors_v1 WHERE buffer_days = %(b)s",
    engine, params={"b": BUFFER_DAYS})

assert len(df) == 19440 and df["hadm_id"].is_unique, "population differs from gate G1"
assert int(df["label_main"].sum()) == 129, "events differ from gate G1"
show("G3 · Data of the gate (7-day margin)", pd.DataFrame([{"n_admissions": len(df), "n_patients": df["subject_id"].nunique(),
                                                             "n_strict_events": int(df["label_main"].sum())}]))

X = design_matrix(df)
y = df["label_main"].astype(int).to_numpy()
groups = df["subject_id"].to_numpy()
P = X.shape[1]
assert P == 27, P
show("G3 · Candidate parameters and missing values", pd.DataFrame([{"candidate parameters": P,
     **{f"missing {k}": int(v) for k, v in X.isna().sum()[X.isna().sum() > 0].items()}}]))

,admisiones,pacientes,eventos estrictos
1,"19,440","13,735",129


,parámetros candidatos,faltantes última sistólica,faltantes última diastólica,faltantes último IMC,faltantes última hemoglobina
1,27,"7,524","7,524","11,136","8,005"


## P3 · Puerta de tamaño de muestra — R² de Cox-Snell observado (validación cruzada) y criterios de Riley et al.

Predicciones fuera de la partición de una regresión logística sin penalización con los 27 parámetros; 5 particiones agrupadas por paciente; imputación múltiple y escalado ajustados con la parte de entrenamiento de cada partición. Un R² negativo significa que, fuera de muestra, las predicciones son peores que asignar a todas las mujeres la misma probabilidad (la prevalencia). Criterios de Riley: encogimiento esperado de 0,9 o más; diferencia de 0,05 o menos entre el R² de Nagelkerke aparente y el ajustado; riesgo global con un margen de ±0,05.

| Resultado | Qué se hace | En este modelo |
|---|---|---|
| Los datos cumplen los tres criterios con el R² observado | P3 pasa: se pasa a E8 (ajuste del modelo) | — |
| No los cumplen, con un R² positivo | Conjunto reducido (17), solo para un modelo que cumple P4, o siguiente peldaño (R1) | — |
| R² observado no positivo | P3 no pasa con ningún número de parámetros: los predictores no tienen información sobre el desenlace | R² = −0,00054: **no pasa** |

In [3]:
# Out-of-fold predictions of an unpenalized logistic regression. Imputation and scaling are
# fitted on the training part of each split only; folds are grouped by patient.
def make_model():
    return make_pipeline(
        IterativeImputer(max_iter=10, random_state=SEED),
        StandardScaler(),
        LogisticRegression(penalty=None, max_iter=5000))

oof = np.full(len(y), np.nan)
cv = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
fold_rows = []
for k, (tr, te) in enumerate(cv.split(X, y, groups), 1):
    m = make_model().fit(X.iloc[tr], y[tr])
    oof[te] = m.predict_proba(X.iloc[te])[:, 1]
    fold_rows.append({"fold": k, "n_admissions": len(te), "n_events": int(y[te].sum())})
assert not np.isnan(oof).any()
show("G3 · Test part of each split", pd.DataFrame(fold_rows))

def loglik(y, p):
    p = np.clip(p, 1e-12, 1 - 1e-12)
    return float(np.sum(y * np.log(p) + (1 - y) * np.log(1 - p)))

n, E = len(y), int(y.sum())
phi = E / n
L0 = n * (phi * np.log(phi) + (1 - phi) * np.log(1 - phi))   # intercept-only model
L1 = loglik(y, oof)
R2_cs = 1 - np.exp(2 * (L0 - L1) / n)
R2_max = 1 - np.exp(2 * L0 / n)

# Riley et al. (BMJ 2020) criteria for a binary outcome
def riley(p, R2, R2max, phi):
    if R2 <= 0:
        return None
    n1 = p / ((S_TARGET - 1) * np.log(1 - R2 / S_TARGET))
    S2 = R2 / (R2 + DELTA * R2max)
    n2 = p / ((S2 - 1) * np.log(1 - R2 / S2))
    n3 = (1.96 / MARGIN) ** 2 * phi * (1 - phi)
    S = max(S_TARGET, S2)
    p_max = int(np.floor(n * (S - 1) * np.log(1 - R2 / S)))   # parameters the data support
    n_req = int(np.ceil(max(n1, n2, n3)))
    return dict(n1=n1, n2=n2, n3=n3, S2=S2, n_required=n_req,
                events_required=int(np.ceil(n_req * phi)), p_supported=p_max)

res = riley(P, R2_cs, R2_max, phi)
if res is None:
    gate = "NOT PASSED (cross-validated R2 is not positive)"
else:
    gate = "PASSED" if n >= res["n_required"] else "NOT PASSED"

# smallest R2 with which the n available admissions would support p parameters (criterion 1, shrinkage 0.9)
r2_needed = lambda p: S_TARGET * (1 - np.exp(p / (n * (S_TARGET - 1))))
show("G3 · Observed Cox-Snell R2 and minimum R2 for the candidate parameters", pd.DataFrame([
    ("prevalence", round(phi, 4)), ("maximum Cox-Snell R2", round(R2_max, 4)),
    ("observed Cox-Snell R2 (cross-validated)", round(R2_cs, 5)),
    ("observed R2, % of the maximum", round(100 * R2_cs / R2_max, 1)),
    (f"minimum R2 for {P} parameters", round(r2_needed(P), 4)),
    (f"minimum R2 for {REDUCED_PARAMETERS} parameters (reduced set)", round(r2_needed(REDUCED_PARAMETERS), 4)),
    ("gate G3", gate)], columns=["indicator", "value"]))
if res is not None:
    show("G3 · Criteria of Riley et al.", pd.Series(res, name="value").to_frame(), index=True)

,partición,admisiones,eventos
1,1,"3,889",26
2,2,"3,887",25
3,3,"3,888",26
4,4,"3,888",26
5,5,"3,888",26


,indicador,valor
1,prevalencia,0.0066
2,R2 de Cox-Snell máximo,0.0767
3,R2 de Cox-Snell observado (validación cruzada),-0.00221
4,"R2 observado, % del máximo",-2.9
5,R2 mínimo para 27 parámetros,0.0124
6,R2 mínimo para 17 parámetros (conjunto reducido),0.0078
7,puerta P3,NO PASA (el R2 con validación cruzada no es positivo)


## P3 · Decisión de la puerta, clase del modelo y reporte (E11)

El conjunto reducido solo se permite para un modelo que ya cumple la puerta P4, y con un R² negativo tampoco pasaría. El retorno R3 ya se usó y la escalera de desenlaces de MIMIC-IV no tiene un peldaño más frecuente para la muerte fetal en esta ruta, así que el modelo queda como **análisis de factibilidad**. No se realizan las etapas E8 a E10 ni la puerta P4.

| Resultado | Qué se hace | En este modelo |
|---|---|---|
| P3 no pasa y no queda retorno ni peldaño | Clase: factibilidad; se reporta la auditoría, la razón del retorno y el resultado de P3 (E11) | **Factibilidad** |
| P3 pasa | Se pasa a E8 (ajuste del modelo) | — |

In [4]:
# Decision of gate G3 and class of M3
passed = gate == "PASSED"
decision("G3 · Sample size gate", [
    ("observed Cox-Snell R2 with 27 parameters", f"{R2_cs:.5f} ({100 * R2_cs / R2_max:.1f}% of the maximum)",
     "passes: on to S8" if passed else "does not pass"),
    ("reduced set of 17 parameters", f"it would need R2 of at least {r2_needed(REDUCED_PARAMETERS):.4f}",
     "not applied: allowed only for a model that meets G4, and with this R2 it would not pass either"),
    ("returns", "R3 already used (admission + 6 h -> antenatal)", "no more frequent rung for fetal death in MIMIC-IV")])
klass = "FEASIBILITY" if not passed else "goes on to S8"
decision("Class of the model and reporting (S11)", [
    ("M3 at admission + 6 h", "stopped at S6: more than half of the deaths already present", "return R3 (notebook 03)"),
    ("M3 · R3 antenatal", "stopped at G3: observed R2 not positive", klass),
    ("stages S8 to S10 and gate G4", "not carried out", "—"),
    ("reporting (S11)", "count audit, reason of the return and result of G3", "with the information MIMIC-IV records before the admission, fetal death cannot be predicted")])

,revisión,resultado,decisión
1,R2 de Cox-Snell observado con 27 parámetros,-0.00221 (-2.9% del máximo),no pasa
2,conjunto reducido de 17 parámetros,necesitaría un R2 de al menos 0.0078,"no se aplica: solo se permite para un modelo que cumple P4, y con este R2 tampoco pasaría"
3,retornos,R3 ya usado (ingreso + 6 h -> prenatal),no hay un peldaño más frecuente para la muerte fetal en MIMIC-IV


,revisión,resultado,decisión
1,M3 al ingreso + 6 h,se detuvo en E6: más de la mitad de las muertes ya presentes,retorno R3 (notebook 03)
2,M3 · R3 prenatal,se detuvo en P3: R2 observado no positivo,FACTIBILIDAD
3,etapas E8 a E10 y puerta P4,no se realizan,—
4,reporte (E11),"auditoría de conteos, razón del retorno y resultado de P3","con la información que MIMIC-IV registra antes del ingreso, la muerte fetal no puede predecirse"


## Resumen del notebook

Indicadores principales de este notebook y el paso siguiente.

In [5]:
# Summary of notebook 06 and of model M3
notebook_summary([
    ("admissions / strict events", lambda: f"{n:,} / {E}"),
    ("candidate parameters", lambda: P),
    ("observed Cox-Snell R2 (cross-validated)", lambda: f"{R2_cs:.5f} ({100 * R2_cs / R2_max:.1f}% of the maximum {R2_max:.4f})"),
    ("minimum R2 for 27 / 17 parameters", lambda: f"{r2_needed(27):.4f} / {r2_needed(17):.4f}"),
    ("gate G3", lambda: gate),
    ("class of M3", lambda: klass),
    ("path of M3", "S1-S6 at admission + 6 h -> return R3 -> S2-S7 and G1, G2 antenatal -> G3 does not pass"),
])
lines_gate = [
    "Gate G3 (sample size), antenatal model; aggregate output only",
    f"buffer_days={BUFFER_DAYS}; folds={N_FOLDS} grouped by patient; seed={SEED}",
    f"admissions={n}; events={E}; prevalence={phi:.4f}; candidate_parameters={P}",
    f"R2_cs_max={R2_max:.4f}; R2_cs_observed_cv={R2_cs:.5f} ({100 * R2_cs / R2_max:.1f}% of max)",
]
if res is not None:
    lines_gate += [f"n_required={res['n_required']} (criterion1={res['n1']:.0f}, criterion2={res['n2']:.0f}, criterion3={res['n3']:.0f}); "
                   f"events_required={res['events_required']}; parameters_supported={res['p_supported']}"]
lines_gate += [f"gate={gate}"]
write_outputs(OUT_FILE, "\n".join(lines_gate))

,valor
indicador,
admisiones / eventos estrictos,"19,440 / 129"
parámetros candidatos,27
R2 de Cox-Snell observado (validación cruzada),-0.00221 (-2.9% del máximo 0.0767)
R2 mínimo para 27 / 17 parámetros,0.0124 / 0.0078
puerta P3,NO PASA (el R2 con validación cruzada no es positivo)
clase del M3,FACTIBILIDAD
ruta del M3,"E1-E6 al ingreso + 6 h -> retorno R3 -> E2-E7 y P1, P2 prenatal -> P3 no pasa"


salida agregada escrita: data/outputs_06_sample_size_gate.txt
